<div style="padding: 20px; background: linear-gradient(90deg, #ee0979 0%, #ff6a00 100%); border-radius: 10px; color: white;">
    <h1 style="color: white; border-bottom: none;">⚖️ Module 3.2: Benchmarking & Selecting Local Models</h1>
    <p style="font-size: 1.2em; opacity: 0.9;">A practical guide to evaluating open-source embedding models for your RAG system.</p>
</div>

---

## 1. The MTEB Leaderboard

The **Massive Text Embedding Benchmark (MTEB)** is the gold standard for evaluating embedding models. It tests models across 8 diverse tasks:

1. **Retrieval** (Crucial for RAG!)
2. **Clustering**
3. **Classification**
4. **Reranking**
5. **STS (Semantic Textual Similarity)**
6. **Summarization**
7. **Pair Classification**
8. **Bitext Mining**

### Cloud vs. Local
Commercial embedding APIs can perform exceptionally well, but many open-source models (like `BGE-M3` or `nomic-embed-text`) offer comparable performance while being **100% free** and preserving complete **data privacy**.

## 2. Comparing Local Models via Python
Let's write a robust script to actually test the speed and vector characteristics of different models available through `sentence-transformers`.

### Course alignment and free-first stack

- Covers: Free local embedding model comparison and cost-quality trade-offs.
- Runtime stack: Groq chat models for generation/evaluation when an LLM is needed, plus local Hugging Face sentence-transformers embeddings for retrieval.
- No paid OpenAI API key is required. Set `GROQ_API_KEY` only for notebooks that call an LLM; pure retrieval and embedding notebooks run locally after model weights are available.
- Current LangChain pattern: provider split packages such as `langchain_groq`, `langchain_huggingface`, and `langchain_chroma`, with runnable `.invoke()` APIs.


In [ ]:
import time
from sentence_transformers import SentenceTransformer
import os

os.environ['TOKENIZERS_PARALLELISM'] = 'false'

# Create a dummy corpus to benchmark processing speed
corpus = [
    "Retrieval-Augmented Generation bridges the gap between LLMs and your data.",
    "Fine-tuning alters the model weights, whereas RAG dynamically injects context.",
    "Embeddings represent the semantic meaning of text in a latent space.",
    "The quick brown fox jumps over the lazy dog.",
    "Artificial Intelligence continues to evolve rapidly in the 21st century."
] * 100 # Multiply by 100 to get a meaningful processing time (500 sentences)

def benchmark_embedding_model(model_id: str):
    print(f"\n--- Benchmarking: {model_id} ---")
    try:
        # 1. Load Model
        start_load = time.time()
        model = SentenceTransformer(model_id)
        load_time = time.time() - start_load
        print(f"[+] Model Loaded in {load_time:.2f} seconds")
        
        # 2. Encode Corpus
        start_encode = time.time()
        embeddings = model.encode(corpus, show_progress_bar=False)
        encode_time = time.time() - start_encode
        
        # 3. Analyze Output
        dim_size = embeddings.shape[1]
        docs_per_sec = len(corpus) / encode_time
        
        print(f"[+] Dimensions Output: {dim_size}")
        print(f"[+] Encoding Time: {encode_time:.2f} seconds for {len(corpus)} documents")
        print(f"[+] Throughput: {docs_per_sec:.2f} documents/second")
        
    except Exception as e:
        print(f"[-] Error loading or running model: {e}")


### Benchmarking Execution
We will test two different models:
1. **all-MiniLM-L6-v2**: Designed specifically for speed and small size.
2. **all-mpnet-base-v2**: Designed for high accuracy (higher dimensions, larger file size).

In [ ]:
# Benchmark 1: The lightweight champion
benchmark_embedding_model('all-MiniLM-L6-v2')

# Benchmark 2: The accuracy champion (slower, but higher quality)
benchmark_embedding_model('all-mpnet-base-v2')

## 3. Vector Dimensionality & Storage Costs

Why not always pick the most accurate, highest-dimension model? **Storage and Search speed.**

If your Vector Database holds 1,000,000 documents:
- `MiniLM` (384 dimensions): `384 * 4 bytes (float32) * 1,000,000` ≈ **1.5 GB** RAM
- `OpenAI` (1536 dimensions): `1536 * 4 bytes * 1,000,000` ≈ **6.1 GB** RAM

Larger vectors mean more RAM usage and slightly slower cosine similarity calculations. For massive enterprise deployments, using a smaller but highly-optimized local model saves thousands of dollars.